In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
df_item = spark.sql(f"""
    SELECT 
        GTIN_CD
        ,ARTICLE_DESC
        ,ARTICLE_NBR
        ,MCH4_CD
        ,MCH4_DESC
        ,MCH3_CD
        ,MCH3_DESC
        ,MCH2_CD
        ,MCH2_DESC
        ,MCH1_CD
        ,MCH1_DESC
        ,MC_CD
        ,MC_DESC
        ,AH1_CD
        ,AH1_DESC
        ,AH2_CD
        ,AH2_DESC
        ,AH3_CD
        ,AH3_DESC
        ,AH4_CD
        ,AH4_DESC
        ,AH5_CD
        ,AH5_DESC
        ,AH6_CD
        ,AH6_DESC
        ,BRAND_TYPE
        ,CAST(EFF_DT AS DATE) AS EFF_DT
        ,CAST(EXP_DT AS DATE) AS EXP_DT
        ,ORDR_AS AS REPLACEMENT_ARTICLE
    FROM 
        {bronze_master_item}
""").dropDuplicates()

df_item.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'item', config_validation, df_item, stats_etl_path
    )

### Merge

In [0]:
df_item.write.mode("overwrite").saveAsTable(silver_master_item)

if archive_flag:
    save_archive(df_item, silver_master_item_archive, run_as_date)